# pkl database → heatmap input CSVs
One run writes one CSV per metric (`Regions, p, r, ...`), ready for `iNPH_heatmap.ipynb`.
Input: the pkl from `analysis.ipynb` (rows = regions; cells = 2×N DataFrames, index `group1`/`group2`; includes a `number of tracts` column).

In [1]:
# CELL 1 - settings (the only cell you normally edit)
from pathlib import Path

PKL_PATH   = "/mnt/1248/open/finley/hydrocephalus/database1.pkl"
OUT_DIR    = "/mnt/1248/open/finley/hydrocephalus/heatmap_inputs"
PREFIX     = "test"          # output name = f"{PREFIX}_{metric}{EXT}"  e.g. QSDR_qa.csv
EXT        = ".csv"          # use "" to match the old extension-less filenames

PATIENT_GROUP = "group2"     # which pkl group is iNPH/Hydro. Patients go first in the test,
                             # so r > 0 means HIGHER in patients (what the heatmap notebook assumes)
CLEANUP_METRIC = None#"number of tracts"
THRESHOLDS = (100, None)     # (min, max) on CLEANUP_METRIC, per subject per region; None = no limit
MIN_N      = 3               # skip a region/metric if either group has fewer subjects than this
METRICS    = None            # None = every column except Regions and CLEANUP_METRIC; or e.g. ["qa", "md"]
BOOTSTRAP_CI = False         # True adds ci_low/ci_high (slow: n_boot resamples per region per metric)
N_BOOT     = 5000

In [2]:
# CELL 2 - pipeline
import pickle
import numpy as np
import pandas as pd
from scipy import stats

def kept_values(cell, tracts, group, lo, hi):
    """Numeric values for one group, keeping only subjects whose tract count is within [lo, hi]."""
    v = pd.to_numeric(cell.loc[group], errors="coerce")
    if tracts is not None:
        t = pd.to_numeric(tracts.loc[group], errors="coerce")
        v = v[t.reindex(v.index).between(lo, hi, inclusive="both")]
    return v.dropna().to_numpy(dtype=float)

def region_stats(pat, ctl, ci=False, n_boot=5000, seed=0):
    u1, p = stats.mannwhitneyu(pat, ctl, alternative="two-sided")
    n1, n2 = len(pat), len(ctl)
    out = {"U1": u1, "Umax": max(u1, n1 * n2 - u1), "p": p,
           "r": 2 * u1 / (n1 * n2) - 1, "n_patients": n1, "n_controls": n2}
    if ci:
        rng = np.random.default_rng(seed)
        rb = lambda a, b: 2 * stats.mannwhitneyu(a, b).statistic / (len(a) * len(b)) - 1
        boot = [rb(rng.choice(pat, n1), rng.choice(ctl, n2)) for _ in range(n_boot)]
        out["ci_low"], out["ci_high"] = np.percentile(boot, [2.5, 97.5])
    return out

def build_tables(db):
    ctl_group = "group1" if PATIENT_GROUP == "group2" else "group2"
    lo = -np.inf if THRESHOLDS[0] is None else THRESHOLDS[0]
    hi = np.inf if THRESHOLDS[1] is None else THRESHOLDS[1]
    metrics = METRICS or [c for c in db.columns if c not in ("Regions", CLEANUP_METRIC)]
    use_clean = CLEANUP_METRIC in db.columns and CLEANUP_METRIC is not None
    tables = {m: [] for m in metrics}
    for _, row in db.iterrows():
        tr = row[CLEANUP_METRIC] if use_clean else None
        for m in metrics:
            pat = kept_values(row[m], tr, PATIENT_GROUP, lo, hi)
            ctl = kept_values(row[m], tr, ctl_group, lo, hi)
            if len(pat) < MIN_N or len(ctl) < MIN_N:
                continue
            tables[m].append({"Regions": row["Regions"],
                              **region_stats(pat, ctl, BOOTSTRAP_CI, N_BOOT)})
    return {m: pd.DataFrame(t) for m, t in tables.items()}

In [3]:
# CELL 3 - run: load pkl, write one CSV per metric
with open(PKL_PATH, "rb") as f:
    db = pickle.load(f)

tables = build_tables(db)
out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)

files = {}
for m, t in tables.items():
    name = f"{PREFIX}_{m}{EXT}"
    t.to_csv(out / name, index=False)
    files[m.upper()] = str(out / name)
    print(f"{m:8s} {len(t):4d} regions -> {name}")

# paste into the FILES dict of iNPH_heatmap.ipynb (edit the labels/order as you like)
print("\nFILES = {")
for k, v in files.items():
    print(f'    "{k}": "{v}",')
print("}")

qa        235 regions -> test_qa.csv
qir       235 regions -> test_qir.csv
vol       235 regions -> test_vol.csv
dti_fa    235 regions -> test_dti_fa.csv
md        235 regions -> test_md.csv
rd        235 regions -> test_rd.csv
iso       235 regions -> test_iso.csv
rdi       235 regions -> test_rdi.csv

FILES = {
    "QA": "/mnt/1248/open/finley/hydrocephalus/heatmap_inputs/test_qa.csv",
    "QIR": "/mnt/1248/open/finley/hydrocephalus/heatmap_inputs/test_qir.csv",
    "VOL": "/mnt/1248/open/finley/hydrocephalus/heatmap_inputs/test_vol.csv",
    "DTI_FA": "/mnt/1248/open/finley/hydrocephalus/heatmap_inputs/test_dti_fa.csv",
    "MD": "/mnt/1248/open/finley/hydrocephalus/heatmap_inputs/test_md.csv",
    "RD": "/mnt/1248/open/finley/hydrocephalus/heatmap_inputs/test_rd.csv",
    "ISO": "/mnt/1248/open/finley/hydrocephalus/heatmap_inputs/test_iso.csv",
    "RDI": "/mnt/1248/open/finley/hydrocephalus/heatmap_inputs/test_rdi.csv",
}
